# Diagnóstico de aprendizagem e lacunas por questionário

## Problema
Professores do Ensino Fundamental I (1º ao 5º ano) não têm um jeito rápido e
padronizado de saber **em quais conteúdos** cada aluno está abaixo do esperado
para a sua série, nem **o que fazer** a respeito.

## Referências
- **BNCC:** define o que se espera que o aluno aprenda em cada ano.
- **SAEB:** avalia Língua Portuguesa e Matemática; por isso o projeto começa por essas duas disciplinas.
- **Compromisso Nacional Criança Alfabetizada (MEC):** a leitura nos anos iniciais continua sendo um dos conteúdos avaliados.

## Quem se beneficia
- **O aluno**, que recebe reforço no conteúdo certo antes de acumular atraso.
- **O professor**, que sabe o que reforçar com cada aluno e com a turma.

## O que a IA faz
1. O aluno responde a um questionário da sua série: **2 disciplinas × 4 conteúdos × 3 questões**.
2. O sistema corrige as respostas e conta os acertos em cada conteúdo.
3. Um modelo de Machine Learning estima, **para cada conteúdo**, a probabilidade de
   haver uma **lacuna** (desempenho abaixo do esperado para a série).
   Trata-se de uma classificação **multirrótulo**: um aluno pode ter nenhuma, uma ou várias lacunas.
4. A partir das lacunas, o sistema gera os **insights**:
   - o **nível em cada disciplina**;
   - os **conteúdos a reforçar**, do mais urgente ao menos urgente;
   - uma **estratégia de estudo** para cada conteúdo, tirada de um catálogo fixo e revisável.

```
respostas ──► validar ──► corrigir ──► 13 features ──► modelo ──► 8 probabilidades
 (série +      (inválido    (acertos       (acertos +      (regressão     de lacuna
  24 letras)    para aqui)   por conteúdo)  série)          logística)        │
                                                                              ▼
     insights ◄── nível por disciplina + conteúdos a reforçar ◄── lacuna / sem_lacuna / a_confirmar
```

Por que o modelo e não só uma regra pelos acertos (como "errou alguma questão do conteúdo")?
Com apenas 3 questões, um chute certo ou um erro de distração mudam o resultado. O modelo usa
também o desempenho nos **outros** conteúdos e a **série** para decidir com mais segurança.
O ganho sobre a melhor regra desse tipo é medido nos passos 9 e 13.

## Conceitos básicos
- **Machine Learning supervisionado:** o modelo aprende a partir de exemplos já rotulados
  (aqui, alunos cujas lacunas o professor já avaliou) e depois prevê o rótulo de alunos novos.
- **Feature:** cada informação de entrada do modelo (aqui, os acertos em cada conteúdo e a série).
- **Rótulo (ou alvo):** o que se quer prever (aqui, lacuna sim/não em cada conteúdo).
- **Classificação binária × multirrótulo:** binária é responder uma pergunta sim/não; multirrótulo
  é responder várias perguntas sim/não ao mesmo tempo para o mesmo aluno (8, uma por conteúdo).
- **Treino e teste:** o modelo aprende com uma parte dos dados (treino) e é avaliado numa parte que
  nunca viu (teste), para medir como ele se sairia com alunos novos.

## Qual erro é o mais grave
**Não detectar uma lacuna que existe:** o aluno ficaria sem o reforço de que precisa.
Apontar uma lacuna que não existe custa menos: uma atividade extra de revisão.

**Métricas** (calculadas por conteúdo e depois com média entre os 8 conteúdos):
- **Prioritária:** recall de lacuna (de todas as lacunas reais, quantas o modelo encontra).
- **De equilíbrio:** F1-macro. Impede que o modelo "ganhe" no recall simplesmente
  apontando lacuna em tudo.

## Uso responsável
- O resultado é uma **triagem**: apoia o professor, não substitui a avaliação dele.
- Questionários inválidos **não recebem diagnóstico**, e conteúdos com resultado
  duvidoso aparecem como **"a confirmar pelo professor"** (falhar fechado).
- As estratégias de estudo vêm de um **catálogo fixo**, não são geradas por IA no momento
  do uso: o texto pode ser revisado por pedagogos e o sistema funciona offline.
- Dados de crianças são sensíveis (LGPD, art. 14). Este projeto usa apenas **dados simulados**.

## ODS
- ODS 4 (Educação de qualidade)
- ODS 10 (Redução das desigualdades)

## Como executar
- **Python 3.11 ou superior** (exigência do pandas 3).
- Instale as dependências: `pip install -r requirements.txt` (de preferência dentro de um ambiente virtual, `python -m venv .venv`).
- Rode `Kernel → Restart & Run All`. As células dependem umas das outras e precisam rodar **em ordem**, de cima para baixo.
- A semente fixa (`SEED`) garante os mesmos resultados em qualquer execução.

**Arquivos gerados pelo notebook:**

| Arquivo | O que é |
|---|---|
| `data/questionarios.csv` | a base simulada (passo 5) |
| `models/modelo.onnx` | o modelo treinado, no formato que o Java lê (passo 16) |
| `models/modelo_info.json` | o contrato: tudo o que o Java precisa saber para usar o modelo (passo 17) |
| `integracao_java/golden_tests.csv` | casos de teste com as respostas esperadas, para conferir o Java (passo 17) |